<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook6_beta_lactam_PKPD_simulation_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6: β-Lactam PK/PD Simulation: Understanding %fT > MIC

This notebook is the fifth module of the interactive clinical pharmacy PK/PD simulation platform.

This section focuses on the core PK/PD index for β-lactam antibacterial agents:

$$
\% fT > MIC
$$

This means:

> The percentage of time during a dosing interval that the free drug concentration remains above the pathogen MIC.

β-Lactam antibiotics usually exhibit time-dependent bacterial killing. Therefore, compared with simply pursuing a very high peak concentration, maintaining the free drug concentration above the MIC for a sufficient duration is usually more important.

This notebook uses a one-compartment intravenous infusion model to help you compare:

- Short infusion
- Extended infusion
- Continuous infusion
- Different MIC values
- Different clearance values, CL
- Different protein binding rates
- Different PK/PD targets

The specific drug example used in this section is **meropenem**. All parameters are used for educational simulation only and should not be used directly as a basis for prescribing in real patients.


## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain why β-lactam antibacterial agents usually focus on time-dependent PK/PD indices.
2. Understand the relationship among MIC, free drug concentration, and protein binding.
3. Calculate \%fT > MIC within a dosing interval.
4. Compare the effects of short infusion, extended infusion, and continuous infusion on \%fT > MIC.
5. Explain why the original dosing regimen may no longer achieve the target when MIC increases.
6. Explain how increased or decreased CL affects β-lactam exposure.
7. Use simulation results to make a preliminary judgment about whether a dosing regimen achieves a predefined PK/PD target.
8. Understand the difference between this simulation and real clinical dosing decisions.


## 2. PK/PD Characteristics of β-Lactam Antibacterial Agents

β-Lactam antibacterial agents include:

- Penicillins
- Cephalosporins
- Carbapenems
- Monobactams
- β-lactam/β-lactamase inhibitor combinations

The antibacterial effect of this drug class is usually most closely related to the following index:

$$
\% fT > MIC
$$

where:

| Symbol | Meaning |
|---|---|
| f | free, the unbound drug fraction |
| T | time |
| MIC | minimum inhibitory concentration |
| %fT > MIC | the percentage of time during a dosing interval that the free drug concentration remains above the MIC |

Only the free drug concentration is generally considered able to directly exert antibacterial activity. Therefore, for drugs with high protein binding, the free concentration should be considered rather than only the total concentration.

$$
C_{free}(t) = C_{total}(t) 	imes (1 - Protein\ Binding)
$$

For example, if the total concentration is 10 mg/L and protein binding is 20%, then the free concentration is:

$$
C_{free} = 10 	imes (1 - 0.20) = 8\ mg/L
$$


## 3. Common PK/PD Targets for β-Lactams

The required PK/PD target may vary across different β-lactam agents, infection sites, patient conditions, and pathogen MIC values.

Common simplified educational targets include:

| Drug class | Common educational target example |
|---|---|
| Penicillins | Approximately 40%–50% fT > MIC |
| Cephalosporins | Approximately 50%–70% fT > MIC |
| Carbapenems | Approximately 40% fT > MIC |
| Severe infection or immunocompromised status | Higher targets may be needed, such as 100% fT > MIC or 100% fT > 4×MIC |

This notebook uses an adjustable target by default so that you can observe target attainment under different target settings.

Note:

> In real clinical practice, PK/PD targets are not fixed. They are influenced by infection severity, infection site, pathogen MIC, host immune status, renal function, tissue penetration, drug safety, and other factors.


## 4. Meropenem as an Educational Example

This notebook uses meropenem as an example of a β-lactam drug.

Meropenem is a carbapenem antibacterial agent and is primarily administered intravenously in clinical practice.

In the official prescribing information, meropenem may be administered by intravenous infusion, with the usual infusion duration generally being 15–30 minutes. Its plasma protein binding is low, approximately 2%. In this notebook, the default protein binding is set to 2% to illustrate the relationship between free and total concentrations.

For educational convenience, this notebook uses a simplified one-compartment intravenous infusion model. The model does not represent all patients and cannot replace the prescribing information, clinical guidelines, therapeutic drug monitoring, or evaluation by a clinical pharmacist.


## 5. One-Compartment Intravenous Infusion Model

For intravenous infusion, if the infusion duration is $T_{inf}$, the infusion rate is:

$$
R_0 = \frac{Dose}{T_{inf}}
$$

During the infusion, the plasma drug concentration can be expressed as:

$$
C(t) = \frac{R_0}{CL} \left(1-e^{-kt}\right)
$$

where:

$$
k = \frac{CL}{V_d}
$$

After the infusion ends, the decline in concentration can be expressed as:

$$
C(t) = C_{end} \cdot e^{-k(t-T_{inf})}
$$

where:

$$
C_{end} = \frac{R_0}{CL}\left(1-e^{-kT_{inf}}\right)
$$

For multiple dosing, the concentration contribution from each infusion can be added together.

This notebook will simulate:

$$
Dose\ q	au\ h
$$

For example:

$$
1000\ mg\ q8h
$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def concentration_single_infusion(t_after_dose, dose_mg, vd_l, cl_l_h, infusion_h):
    """
    Concentration contribution from one IV infusion dose in a one-compartment model.
    """
    k_elim = cl_l_h / vd_l
    infusion_h = max(infusion_h, 1e-6)
    rate_mg_h = dose_mg / infusion_h

    concentration = np.zeros_like(t_after_dose, dtype=float)

    during = (t_after_dose >= 0) & (t_after_dose <= infusion_h)
    after = t_after_dose > infusion_h

    concentration[during] = (
        rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * t_after_dose[during]))
    )

    c_end = rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * infusion_h))
    concentration[after] = c_end * np.exp(-k_elim * (t_after_dose[after] - infusion_h))

    return concentration


def concentration_multiple_infusions(t, dose_mg, vd_l, cl_l_h, tau_h, infusion_h, n_doses):
    """
    Concentration-time profile after repeated intermittent IV infusions.
    """
    concentration = np.zeros_like(t, dtype=float)
    dose_times = np.arange(n_doses) * tau_h

    for dose_time in dose_times:
        t_after_dose = t - dose_time
        concentration += concentration_single_infusion(
            t_after_dose=t_after_dose,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            infusion_h=infusion_h
        )

    return concentration


def concentration_continuous_infusion(t, daily_dose_mg, vd_l, cl_l_h, loading_dose_mg=0):
    """
    Concentration-time profile during continuous infusion with optional loading dose.
    """
    k_elim = cl_l_h / vd_l
    rate_mg_h = daily_dose_mg / 24

    infusion_component = rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * t))
    loading_component = (loading_dose_mg / vd_l) * np.exp(-k_elim * t)

    concentration = infusion_component + loading_component
    return concentration


def free_concentration(total_concentration, protein_binding_percent):
    """
    Convert total concentration to free concentration.
    """
    free_fraction = 1 - protein_binding_percent / 100
    return total_concentration * free_fraction


def percent_time_above_threshold(t, concentration, threshold, start_time, end_time):
    """
    Calculate percentage of time concentration is above a threshold during a time window.
    """
    mask = (t >= start_time) & (t <= end_time)
    if np.sum(mask) < 2:
        return np.nan

    t_window = t[mask]
    c_window = concentration[mask]
    above = (c_window > threshold).astype(float)

    duration = t_window[-1] - t_window[0]
    if duration <= 0:
        return np.nan

    percent = np.trapz(above, t_window) / duration * 100
    return percent


def calculate_interval_metrics(t, total_conc, free_conc, mic, start_time, end_time):
    """
    Calculate PK/PD metrics in a selected dosing interval.
    """
    mask = (t >= start_time) & (t <= end_time)
    t_window = t[mask]
    total_window = total_conc[mask]
    free_window = free_conc[mask]

    ft_mic = percent_time_above_threshold(
        t=t,
        concentration=free_conc,
        threshold=mic,
        start_time=start_time,
        end_time=end_time
    )

    metrics = {
        "Total Cmax": np.max(total_window),
        "Total Cmin": np.min(total_window),
        "Free Cmax": np.max(free_window),
        "Free Cmin": np.min(free_window),
        "%fT>MIC": ft_mic,
        "Interval start": start_time,
        "Interval end": end_time
    }

    return metrics

## 6. Interactive Simulation 1: Multiple-Dose Infusion and %fT > MIC

The simulation below shows how plasma drug concentration changes over time after multiple intravenous infusions.

Focus on the following:

- Total concentration curve
- Free concentration curve
- MIC horizontal line
- Percentage of time that the free concentration remains above the MIC
- Whether the target \%fT > MIC is achieved within the dosing interval

In this simulation, you can adjust:

- Dose: dose per administration
- Vd: apparent volume of distribution
- CL: clearance
- Tau: dosing interval
- Infusion: infusion duration
- Protein binding: protein binding rate
- MIC: minimum inhibitory concentration
- Target: predefined PK/PD target


In [ ]:
def plot_beta_lactam_pkpd(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    infusion_h=0.5,
    n_doses=6,
    protein_binding_percent=2,
    mic=2,
    target_percent=40
):
    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 2500)

    total_conc = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    free_conc = free_concentration(total_conc, protein_binding_percent)

    last_start = (n_doses - 1) * tau_h
    last_end = n_doses * tau_h

    metrics = calculate_interval_metrics(
        t=t,
        total_conc=total_conc,
        free_conc=free_conc,
        mic=mic,
        start_time=last_start,
        end_time=last_end
    )

    target_achieved = metrics["%fT>MIC"] >= target_percent

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, total_conc, linewidth=2, label="Total concentration")
    ax.plot(t, free_conc, linewidth=2, linestyle="--", label="Free concentration")
    ax.axhline(mic, linestyle=":", linewidth=2, label=f"MIC = {mic:.2f} mg/L")

    ax.fill_between(
        t,
        free_conc,
        mic,
        where=free_conc > mic,
        alpha=0.15,
        interpolate=True,
        label="Free concentration above MIC"
    )

    ax.axvspan(last_start, last_end, alpha=0.08, label="Last dosing interval")

    ax.set_title("Multiple-Dose Beta-Lactam PK/PD Simulation")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend(loc="upper right")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Dose",
            "Dosing interval",
            "Infusion duration",
            "Vd",
            "CL",
            "Protein binding",
            "MIC",
            "Target %fT>MIC",
            "Last-interval %fT>MIC",
            "Target achieved",
            "Total Cmax in last interval",
            "Total Cmin in last interval",
            "Free Cmax in last interval",
            "Free Cmin in last interval"
        ],
        "Value": [
            f"{dose_mg:.0f} mg",
            f"q{tau_h:.1f}h",
            f"{infusion_h:.2f} h",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.1f} L/h",
            f"{protein_binding_percent:.1f}%",
            f"{mic:.2f} mg/L",
            f"{target_percent:.0f}%",
            f"{metrics['%fT>MIC']:.1f}%",
            "Yes" if target_achieved else "No",
            f"{metrics['Total Cmax']:.2f} mg/L",
            f"{metrics['Total Cmin']:.2f} mg/L",
            f"{metrics['Free Cmax']:.2f} mg/L",
            f"{metrics['Free Cmin']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    plot_beta_lactam_pkpd,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    infusion_h=FloatSlider(value=0.5, min=0.25, max=8, step=0.25, description="Infusion"),
    n_doses=IntSlider(value=6, min=2, max=12, step=1, description="Doses"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    mic=FloatSlider(value=2, min=0.25, max=16, step=0.25, description="MIC"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target")
);

## 7. Observation Task 1: Which Factors Influence %fT > MIC?

Use the interactive simulation above to complete the following tasks.

### Task A: Standard Regimen

Set:

- Dose = 1000 mg
- Vd = 20 L
- CL = 10 L/h
- Tau = 8 h
- Infusion = 0.5 h
- Protein binding = 2%
- MIC = 2 mg/L
- Target = 40%

Record:

- Last-interval %fT > MIC
- Total Cmax
- Total Cmin
- Free Cmax
- Free Cmin
- Whether the target is achieved

### Task B: Increased MIC

Change only MIC to 4 mg/L or 8 mg/L.

Observe:

- Does %fT > MIC decrease?
- Does the original regimen still achieve the target?
- Why can the same dose produce different efficacy at different MIC values?

### Task C: Increased Clearance

Change CL to 20 L/h.

Observe:

- Does the concentration decline faster?
- Does %fT > MIC decrease?
- What clinical situation can this scenario simulate?

Hint: Some critically ill patients may develop augmented renal clearance, leading to β-lactam concentrations below expectations.

### Task D: Decreased Clearance

Change CL to 4 L/h.

Observe:

- Is the concentration maintained for a longer time?
- Does %fT > MIC increase?
- Does a higher value always mean better?

Hint: Decreased CL may increase the probability of target attainment, but it may also increase the risk of adverse effects, especially in patients with renal impairment.


## 8. Short Infusion, Extended Infusion, and Continuous Infusion

The core goal for β-lactam antibiotics is to keep the free concentration above the MIC for a sufficient duration.

Therefore, changing the infusion strategy may substantially affect \%fT > MIC.

Common administration strategies include:

| Administration strategy | Characteristics |
|---|---|
| Short infusion | Short infusion duration and higher peak concentration, but concentration declines relatively quickly |
| Extended infusion | Longer infusion duration and possibly lower peak concentration, but the time above MIC may be prolonged |
| Continuous infusion | Continuous administration, aiming to maintain a relatively stable concentration |

Note:

> Extended infusion or continuous infusion is not required in every situation. Whether to use it should be considered together with drug stability, infection severity, MIC, renal function, nursing feasibility, and institutional protocols.


In [ ]:
def compare_infusion_strategies(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    short_infusion_h=0.5,
    extended_infusion_h=3,
    protein_binding_percent=2,
    mic=2,
    target_percent=40,
    loading_dose_mg=0
):
    t_end_h = 48
    t = np.linspace(0, t_end_h, 3000)
    n_doses = int(np.floor(t_end_h / tau_h))
    daily_dose_mg = dose_mg * (24 / tau_h)

    short_total = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=short_infusion_h,
        n_doses=n_doses
    )

    extended_total = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=extended_infusion_h,
        n_doses=n_doses
    )

    continuous_total = concentration_continuous_infusion(
        t=t,
        daily_dose_mg=daily_dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        loading_dose_mg=loading_dose_mg
    )

    short_free = free_concentration(short_total, protein_binding_percent)
    extended_free = free_concentration(extended_total, protein_binding_percent)
    continuous_free = free_concentration(continuous_total, protein_binding_percent)

    last_start = t_end_h - tau_h
    last_end = t_end_h
    last_24_start = t_end_h - 24

    short_metrics = calculate_interval_metrics(t, short_total, short_free, mic, last_start, last_end)
    extended_metrics = calculate_interval_metrics(t, extended_total, extended_free, mic, last_start, last_end)

    continuous_ft = percent_time_above_threshold(
        t=t,
        concentration=continuous_free,
        threshold=mic,
        start_time=last_24_start,
        end_time=t_end_h
    )

    continuous_mask = (t >= last_24_start) & (t <= t_end_h)
    continuous_metrics = {
        "Total Cmax": np.max(continuous_total[continuous_mask]),
        "Total Cmin": np.min(continuous_total[continuous_mask]),
        "Free Cmax": np.max(continuous_free[continuous_mask]),
        "Free Cmin": np.min(continuous_free[continuous_mask]),
        "%fT>MIC": continuous_ft
    }

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, short_free, linewidth=2, label=f"Short infusion ({short_infusion_h:.1f} h)")
    ax.plot(t, extended_free, linewidth=2, label=f"Extended infusion ({extended_infusion_h:.1f} h)")
    ax.plot(t, continuous_free, linewidth=2, label="Continuous infusion")
    ax.axhline(mic, linestyle=":", linewidth=2, label=f"MIC = {mic:.2f} mg/L")
    ax.axvspan(last_start, last_end, alpha=0.08, label="Last intermittent interval")

    ax.set_title("Short vs Extended vs Continuous Infusion")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Free concentration (mg/L)")
    ax.legend(loc="upper right")
    plt.show()

    summary = pd.DataFrame({
        "Strategy": ["Short infusion", "Extended infusion", "Continuous infusion"],
        "Infusion duration": [
            f"{short_infusion_h:.2f} h",
            f"{extended_infusion_h:.2f} h",
            "24 h"
        ],
        "Total daily dose": [
            f"{daily_dose_mg:.0f} mg/day",
            f"{daily_dose_mg:.0f} mg/day",
            f"{daily_dose_mg:.0f} mg/day"
        ],
        "%fT>MIC": [
            f"{short_metrics['%fT>MIC']:.1f}%",
            f"{extended_metrics['%fT>MIC']:.1f}%",
            f"{continuous_metrics['%fT>MIC']:.1f}%"
        ],
        "Target achieved": [
            "Yes" if short_metrics["%fT>MIC"] >= target_percent else "No",
            "Yes" if extended_metrics["%fT>MIC"] >= target_percent else "No",
            "Yes" if continuous_metrics["%fT>MIC"] >= target_percent else "No"
        ],
        "Free Cmax": [
            f"{short_metrics['Free Cmax']:.2f} mg/L",
            f"{extended_metrics['Free Cmax']:.2f} mg/L",
            f"{continuous_metrics['Free Cmax']:.2f} mg/L"
        ],
        "Free Cmin": [
            f"{short_metrics['Free Cmin']:.2f} mg/L",
            f"{extended_metrics['Free Cmin']:.2f} mg/L",
            f"{continuous_metrics['Free Cmin']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    compare_infusion_strategies,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    short_infusion_h=FloatSlider(value=0.5, min=0.25, max=2, step=0.25, description="Short"),
    extended_infusion_h=FloatSlider(value=3, min=1, max=8, step=0.5, description="Extended"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    mic=FloatSlider(value=2, min=0.25, max=16, step=0.25, description="MIC"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target"),
    loading_dose_mg=FloatSlider(value=0, min=0, max=3000, step=250, description="Loading")
);

## 9. Observation Task 2: Why May Extended Infusion Improve Target Attainment?

Use the comparison simulation above to complete the following tasks.

### Task A: Compare Three Infusion Strategies

Set:

- Dose = 1000 mg
- Tau = 8 h
- Short infusion = 0.5 h
- Extended infusion = 3 h
- CL = 10 L/h
- Vd = 20 L
- MIC = 2 mg/L
- Target = 40%

Compare:

- %fT > MIC with short infusion
- %fT > MIC with extended infusion
- %fT > MIC with continuous infusion

Think about:

> Why may extended infusion produce a lower peak concentration but still increase %fT > MIC?

### Task B: Increased MIC

Change MIC to 4 mg/L or 8 mg/L.

Observe:

- Which infusion strategy is more likely to maintain the concentration above the MIC?
- Could the original short-infusion regimen fail to achieve the target?

### Task C: Same Total Daily Dose

In this comparison, short infusion, extended infusion, and continuous infusion use the same total daily dose.

Think about:

> When the total daily dose is the same, why can changing only the infusion strategy alter PK/PD target attainment?


## 10. MIC Sensitivity Analysis

MIC is an important parameter linking drug exposure to antibacterial effect.

For the same dosing regimen, the target may be achieved when MIC is low; when MIC increases, the target may no longer be achieved.

The simulation below fixes the dosing regimen and then observes \%fT > MIC at different MIC values.


In [ ]:
def mic_sensitivity_analysis(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    infusion_h=3,
    n_doses=6,
    protein_binding_percent=2,
    target_percent=40
):
    mic_values = np.array([0.25, 0.5, 1, 2, 4, 8, 16])
    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 2500)

    total_conc = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    free_conc = free_concentration(total_conc, protein_binding_percent)

    last_start = (n_doses - 1) * tau_h
    last_end = n_doses * tau_h

    ft_values = []
    for mic in mic_values:
        ft = percent_time_above_threshold(
            t=t,
            concentration=free_conc,
            threshold=mic,
            start_time=last_start,
            end_time=last_end
        )
        ft_values.append(ft)

    fig, ax = plt.subplots()
    ax.plot(mic_values, ft_values, marker="o", linewidth=2)
    ax.axhline(target_percent, linestyle="--", label=f"Target = {target_percent:.0f}%")
    ax.set_xscale("log", base=2)
    ax.set_xticks(mic_values)
    ax.set_xticklabels([str(x) for x in mic_values])
    ax.set_title("MIC Sensitivity Analysis")
    ax.set_xlabel("MIC (mg/L)")
    ax.set_ylabel("%fT>MIC in last interval")
    ax.set_ylim(0, 105)
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "MIC (mg/L)": mic_values,
        "%fT>MIC": [f"{x:.1f}%" for x in ft_values],
        "Target achieved": ["Yes" if x >= target_percent else "No" for x in ft_values]
    })

    display(summary)


interact(
    mic_sensitivity_analysis,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    infusion_h=FloatSlider(value=3, min=0.25, max=8, step=0.25, description="Infusion"),
    n_doses=IntSlider(value=6, min=2, max=12, step=1, description="Doses"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target")
);

## 11. Observation Task 3: Relationship Between MIC and Probability of Treatment Success

Use the MIC sensitivity analysis above to complete the following tasks.

### Task A: Default Regimen

Set:

- Dose = 1000 mg
- Tau = 8 h
- Infusion = 3 h
- CL = 10 L/h
- Vd = 20 L
- Protein binding = 2%
- Target = 40%

Observe:

- Is the target achieved when MIC = 1 mg/L?
- Is the target achieved when MIC = 4 mg/L?
- Is the target achieved when MIC = 8 mg/L?

### Task B: Increase the Target

Change Target from 40% to 100%.

Observe:

- At which MIC values is the target still achieved?
- Why might patients with severe infections or immunocompromised status require higher targets?

### Task C: Change CL

Change CL to 20 L/h.

Observe:

- At the same MIC, does %fT > MIC decrease?
- Why might patients with augmented clearance require more aggressive dose optimization?


## 12. Effect of Protein Binding on Free Concentration

Protein binding varies greatly among β-lactam drugs.

For drugs with low protein binding, total concentration and free concentration are not very different.  
For drugs with high protein binding, total concentration may appear adequate, but free concentration may be substantially lower than total concentration.

This section observes the following relationship by changing protein binding:

$$
C_{free}(t) = C_{total}(t) 	imes (1 - Protein\ Binding)
$$

The higher the protein binding, the lower the free concentration.


In [ ]:
def protein_binding_demo(
    total_concentration=20,
    mic=4
):
    binding_values = np.arange(0, 96, 5)
    free_values = total_concentration * (1 - binding_values / 100)

    fig, ax = plt.subplots()
    ax.plot(binding_values, free_values, marker="o", linewidth=2)
    ax.axhline(mic, linestyle="--", label=f"MIC = {mic:.1f} mg/L")
    ax.set_title("Effect of Protein Binding on Free Concentration")
    ax.set_xlabel("Protein binding (%)")
    ax.set_ylabel("Free concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Protein binding (%)": binding_values,
        "Free concentration (mg/L)": np.round(free_values, 2),
        "Free concentration > MIC": ["Yes" if x > mic else "No" for x in free_values]
    })

    display(summary)


interact(
    protein_binding_demo,
    total_concentration=FloatSlider(value=20, min=1, max=100, step=1, description="Total C"),
    mic=FloatSlider(value=4, min=0.25, max=32, step=0.25, description="MIC")
);

## 13. Observation Task 4: Why Should We Focus on Free Concentration?

Use the protein binding simulation above to complete the following tasks.

### Task A: Low Protein Binding

Set:

- Total concentration = 20 mg/L
- MIC = 4 mg/L
- Observe protein binding from 0% to 20%

Think about:

- Is the free concentration close to the total concentration?
- Is it likely to remain above the MIC?

### Task B: High Protein Binding

Observe protein binding from 80% to 95%.

Think about:

- With the same total concentration, does the free concentration decrease substantially?
- If only total concentration is considered, could effective exposure be overestimated?

### Task C: Clinical Connection

Think about:

> How might hypoalbuminemia, critical illness, or changes in renal function affect the free concentration and clearance of highly protein-bound β-lactam drugs?


## 14. Scope and Limitations of the Model in This Notebook

This notebook is intended for education and is not a clinical prescribing system.

This simulation makes the following simplifications:

1. A one-compartment model is used to describe all β-lactam drugs.
2. CL and Vd are assumed to remain constant during the simulation.
3. First-order elimination is assumed.
4. Free concentration is calculated using a fixed protein binding rate.
5. Plasma concentration is used instead of infection-site concentration.
6. \%fT > MIC is used as the main PD index.
7. Real patient condition, infection site, pathogen burden, immune status, and drug toxicity are not incorporated.

In real clinical practice, β-lactam dosing regimens need to be considered together with:

- Prescribing information
- Infectious disease treatment guidelines
- Antimicrobial susceptibility results
- MIC
- Renal function
- Infection site
- Severity of the patient's condition
- Drug stability and infusion feasibility
- Therapeutic drug monitoring when necessary


## 15. Self-Assessment: β-Lactam PK/PD

Complete the following self-assessment questions based on this notebook. It is recommended that you answer independently first and then check the reference answers in the next cell.

---

### Question 1: Which PK/PD index is most commonly used for β-lactam antibacterial agents?

A. Cmax / MIC  
B. AUC / MIC  
C. %fT > MIC  
D. Tmax / MIC  

---

### Question 2: Why is free concentration usually considered when calculating β-lactam PK/PD?

A. Because protein-bound drug usually cannot directly exert antibacterial activity  
B. Because total concentration is always equal to free concentration  
C. Because the higher the protein binding, the higher the free concentration must be  
D. Because MIC applies only to oral drugs  

---

### Question 3: When the total daily dose is the same, what change may extended infusion produce?

A. It always makes AUC become 0  
B. It may prolong the time that the free concentration remains above the MIC  
C. It always lowers the MIC  
D. It always lowers CL  

---

### Question 4: Under the same dosing regimen, what does an increase in MIC usually lead to?

A. Increased %fT > MIC  
B. Decreased %fT > MIC  
C. Protein binding becomes 0  
D. Drug clearance automatically decreases  

---

### Question 5: When CL increases, how is the β-lactam concentration curve most likely to change?

A. Drug elimination becomes slower, and concentration is maintained longer  
B. Drug elimination becomes faster, and %fT > MIC may decrease  
C. MIC must decrease  
D. Vd must become 0


## 16. Self-Assessment Reference Answers

### Question 1

**Reference answer: C**

**Explanation:**  
β-Lactams usually exhibit time-dependent antibacterial activity. The commonly used PK/PD index is:

$$
\% fT > MIC
$$

This is the percentage of time during a dosing interval that the free drug concentration remains above the MIC.

---

### Question 2

**Reference answer: A**

**Explanation:**  
The unbound free drug fraction is generally considered more able to distribute directly and exert antibacterial activity. Therefore, when calculating β-lactam PK/PD indices, the following should be considered:

$$
C_{free}(t) = C_{total}(t) 	imes (1 - Protein\ Binding)
$$

---

### Question 3

**Reference answer: B**

**Explanation:**  
Extended infusion increases the duration of drug input. Although peak concentration may be lower, the time that the free concentration remains above the MIC may increase, which may improve \%fT > MIC.

---

### Question 4

**Reference answer: B**

**Explanation:**  
The higher the MIC, the harder it is for the drug concentration to exceed the MIC. Under the same dosing regimen, an increase in MIC usually decreases \%fT > MIC and may cause the original regimen to fail to achieve the target.

---

### Question 5

**Reference answer: B**

**Explanation:**  
An increase in CL means faster drug clearance. When other parameters remain unchanged, concentration declines faster, and the time that free concentration remains above the MIC may shorten. Therefore, \%fT > MIC may decrease.


## 17. Notebook Summary

This notebook introduced the basic PK/PD logic of β-lactam antibiotics using a one-compartment intravenous infusion model.

Key takeaways:

1. β-Lactam antibiotics are typically evaluated using time-dependent PK/PD indices.
2. The key PK/PD index is %fT > MIC, which reflects how long free drug concentration remains above the MIC.
3. Free concentration is more directly related to antibacterial activity than total concentration.
4. Higher MIC values make it harder for the same regimen to achieve the PK/PD target.
5. Increased clearance can lower exposure and reduce %fT > MIC.
6. Extended or continuous infusion may improve target attainment, but clinical feasibility and drug stability must also be considered.

The complete logic of this section can be summarized as:

$$
Dose + Infusion\ Strategy + CL + V_d \rightarrow C_{free}(t) \rightarrow \%fT > MIC \rightarrow PK/PD\ Target\ Attainment
$$

The next section will further study:

> Aminoglycoside PK/PD simulation: Cmax/MIC, peak concentration, and trough concentration.


## 18. Key References

1. Osthoff M, Siegemund M, Balestra G, Abdul-Aziz MH, Roberts JA. **Prolonged administration of β-lactam antibiotics: a comprehensive review and critical appraisal.** Swiss Medical Weekly. 2016;146:w14368. DOI: 10.4414/smw.2016.14368.
2. DailyMed. **Meropenem for Injection, powder, for solution.** U.S. National Library of Medicine. The prescribing information includes meropenem intravenous infusion, protein binding, and pharmacokinetic information.
3. Infectious Diseases Society of America. **IDSA 2024 Guidance on the Treatment of Antimicrobial Resistant Gram-Negative Infections.** This guidance discusses the use of extended-infusion β-lactam antibiotics in selected scenarios.

The parameters and scenarios in this notebook are used for educational demonstration and should not be used for individualized prescribing in real patients.
